# UpsureAI — YOLOv8n Damage Detector Training
**Run on Google Colab with T4 GPU (~30-40 min)**

## Before running:
1. **Runtime → Change runtime type → T4 GPU**
2. Upload `cardd_roboflow.zip` to `MyDrive/UpSureAI/cardd_roboflow.zip`
3. Run all cells top to bottom

## What this trains:
- YOLOv8n fine-tuned on CarDD (Roboflow v3) — 2000 images, 11 damage classes
- Classes: car-part-crack, detachment, flat-tire, glass-crack, lamp-crack,
  minor-deformation, moderate-deformation, paint-chips, scratches, severe-deformation, side-mirror-crack
- Output: `damage_yolo.onnx` for CPU inference in production API

In [ ]:
# ── CONFIGURE THIS ─────────────────────────────────────────────────────────────
DRIVE_FOLDER = 'MyDrive/UpSureAI'   # Drive folder path
MODEL_SIZE   = 'yolov8n'             # yolov8n (fastest) or yolov8s (better mAP)
EPOCHS       = 50
IMG_SIZE     = 640
BATCH        = 16
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics onnx onnxscript -q

In [ ]:
import shutil, yaml, os
from pathlib import Path

DRIVE_BASE = Path(f'/content/drive/{DRIVE_FOLDER}')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

WORK_DIR          = Path('/content/CarDD-3')
DRIVE_DATASET_ZIP = DRIVE_BASE / 'cardd_roboflow.zip'
DRIVE_OUT         = DRIVE_BASE / 'yolo_output'
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    'car-part-crack', 'detachment', 'flat-tire', 'glass-crack',
    'lamp-crack', 'minor-deformation', 'moderate-deformation',
    'paint-chips', 'scratches', 'severe-deformation', 'side-mirror-crack'
]

assert DRIVE_DATASET_ZIP.exists(), f"Not found: {DRIVE_DATASET_ZIP} — upload cardd_roboflow.zip to {DRIVE_BASE}"

if not WORK_DIR.exists():
    print('Unzipping...')
    shutil.unpack_archive(str(DRIVE_DATASET_ZIP), '/content')
    print('Done.')
else:
    print('Already extracted — skipping.')

for split in ['train', 'valid', 'test']:
    imgs = list((WORK_DIR / split / 'images').glob('*'))
    print(f'  {split}: {len(imgs)} images')

In [ ]:
import yaml

# Roboflow data.yaml uses relative paths — write absolute version for Colab
data_yaml = {
    'path'  : str(WORK_DIR),
    'train' : 'train/images',
    'val'   : 'valid/images',
    'test'  : 'test/images',
    'nc'    : len(CLASS_NAMES),
    'names' : CLASS_NAMES,
}

yaml_path = WORK_DIR / 'data_colab.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print('data_colab.yaml written:')
print(open(yaml_path).read())

In [ ]:
from ultralytics import YOLO
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

model = YOLO(f'{MODEL_SIZE}.pt')

results = model.train(
    data    = str(yaml_path),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH,
    device  = 0 if torch.cuda.is_available() else 'cpu',
    project = '/content/runs',
    name    = 'damage_yolo',
    exist_ok= True,
    patience= 15,
    save    = True,
    plots   = True,
    verbose = True,
)

print('\nTraining complete.')
print(f'Best weights: {results.save_dir}/weights/best.pt')

In [ ]:
best_pt = Path('/content/runs/damage_yolo/weights/best.pt')
model   = YOLO(best_pt)

metrics = model.val(data=str(yaml_path), split='test')
print('\nTEST METRICS')
print(f'mAP50     : {metrics.box.map50:.4f}')
print(f'mAP50-95  : {metrics.box.map:.4f}')
print('\nPer-class mAP50:')
for name, ap in zip(CLASS_NAMES, metrics.box.ap50):
    print(f'  {name:<15} {ap:.4f}')

In [ ]:
onnx_path = model.export(
    format  = 'onnx',
    opset   = 17,
    dynamic = True,
    simplify= True,
)
print(f'ONNX exported: {onnx_path}')

In [ ]:
import shutil
from google.colab import files

onnx_src = Path(onnx_path)
pt_src   = best_pt

shutil.copy(onnx_src, DRIVE_OUT / 'damage_yolo.onnx')
shutil.copy(pt_src,   DRIVE_OUT / 'damage_yolo_best.pt')

# Copy training plots
run_dir = Path('/content/runs/damage_yolo')
for plot in run_dir.glob('*.png'):
    shutil.copy(plot, DRIVE_OUT / plot.name)

print(f'Saved to Drive: {DRIVE_OUT}')

files.download(str(onnx_src))
print('Download started.')
print('Save to: C:\\Users\\sriva\\Desktop\\UpsureAI\\models\\damage_yolo.onnx')